In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
load_dotenv()
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=""
)

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling crusty and needed to work through some dough-y issues.',
 'explanation': 'This joke is a play on words, using puns to create humor. The joke relies on multiple layers of wordplay related to pizza and baking.\n\n- "Crusty" is a common term for the outside layer of a pizza, but it\'s also a colloquialism for being irritable or having a bad mood.\n- "Dough-y issues" is a pun on "dough" (the mixture of flour, water, and other ingredients used to make pizza crust) and "doughy" (a colloquialism for something that is emotional or psychological, similar to "touchy-feely" issues).\n- "Work through" implies that the pizza is seeking help to resolve its emotional problems, which adds to the therapeutic context.\n\nThe joke is funny because it takes a common phrase related to therapy ("work through some issues") and replaces it with pizza-related puns, creating a clever and unexpected connection between the se

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling crusty and needed to work through some dough-y issues.', 'explanation': 'This joke is a play on words, using puns to create humor. The joke relies on multiple layers of wordplay related to pizza and baking.\n\n- "Crusty" is a common term for the outside layer of a pizza, but it\'s also a colloquialism for being irritable or having a bad mood.\n- "Dough-y issues" is a pun on "dough" (the mixture of flour, water, and other ingredients used to make pizza crust) and "doughy" (a colloquialism for something that is emotional or psychological, similar to "touchy-feely" issues).\n- "Work through" implies that the pizza is seeking help to resolve its emotional problems, which adds to the therapeutic context.\n\nThe joke is funny because it takes a common phrase related to therapy ("work through some issues") and replaces it with pizza-related puns, creating a clever and unexpected connec

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling crusty and needed to work through some dough-y issues.', 'explanation': 'This joke is a play on words, using puns to create humor. The joke relies on multiple layers of wordplay related to pizza and baking.\n\n- "Crusty" is a common term for the outside layer of a pizza, but it\'s also a colloquialism for being irritable or having a bad mood.\n- "Dough-y issues" is a pun on "dough" (the mixture of flour, water, and other ingredients used to make pizza crust) and "doughy" (a colloquialism for something that is emotional or psychological, similar to "touchy-feely" issues).\n- "Work through" implies that the pizza is seeking help to resolve its emotional problems, which adds to the therapeutic context.\n\nThe joke is funny because it takes a common phrase related to therapy ("work through some issues") and replaces it with pizza-related puns, creating a clever and unexpected conne

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and had a lot of "noodle" issues to work through.',
 'explanation': 'The joke is a play on words, using puns to create a humorous connection between the concept of spaghetti (a type of pasta) and the idea of psychological therapy.\n\nThe phrase "feeling a little \'twisted\'" is a pun on the fact that spaghetti is a type of twisted or curved pasta. However, in a psychological context, "twisted" can also mean being disturbed or having an abnormal thought pattern. The joke cleverly uses this double meaning to create a connection between the physical characteristics of spaghetti and the emotional state of needing therapy.\n\nThe second part of the joke, "had a lot of \'noodle\' issues to work through," is another pun. In this case, "noodle" has a double meaning - it refers to the type of pasta (noodles) and also sounds similar to "nuddle," which is a play on the word "nuddle," me

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and had a lot of "noodle" issues to work through.', 'explanation': 'The joke is a play on words, using puns to create a humorous connection between the concept of spaghetti (a type of pasta) and the idea of psychological therapy.\n\nThe phrase "feeling a little \'twisted\'" is a pun on the fact that spaghetti is a type of twisted or curved pasta. However, in a psychological context, "twisted" can also mean being disturbed or having an abnormal thought pattern. The joke cleverly uses this double meaning to create a connection between the physical characteristics of spaghetti and the emotional state of needing therapy.\n\nThe second part of the joke, "had a lot of \'noodle\' issues to work through," is another pun. In this case, "noodle" has a double meaning - it refers to the type of pasta (noodles) and also sounds similar to "nuddle," which is a play on th